# KLA Hackathon - Data Exploration

Notebook for exploring the semiconductor image restoration dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from data.degrade import synthetic_degrade, create_degradation_pipeline
from data.dataset import SemiconductorDataset, create_dataloaders

%matplotlib inline

## 1. Check Data Structure

In [ ]:
data_root = Path('../data/train')

if data_root.exists():
    deg_files = list((data_root / 'degraded').glob('*'))
    gt_files = list((data_root / 'ground_truth').glob('*'))
    print(f"Degraded images: {len(deg_files)}")
    print(f"Ground truth images: {len(gt_files)}")
    
    # Check first few
    for f in deg_files[:5]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        print(f"  {f.name}: {img.shape}, range=[{img.min()}, {img.max()}]")
else:
    print("Data not found. Place data in ../data/train/{degraded,ground_truth}/")

## 2. Visualize Synthetic Degradation

In [ ]:
# Create synthetic ground truth (semiconductor-like pattern)
gt = np.zeros((512, 512), dtype=np.float32)
cv2.line(gt, (0, 256), (512, 256), 1.0, 2)
cv2.line(gt, (256, 0), (256, 512), 1.0, 2)
cv2.rectangle(gt, (100, 100), (200, 200), 0.8, -1)
cv2.circle(gt, (400, 400), 50, 0.6, -1)
cv2.ellipse(gt, (150, 400), (30, 60), 45, 0, 360, 0.7, -1)
gt = cv2.GaussianBlur(gt, (5, 5), 1.0)

# Apply different degradations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gt, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Ground Truth (512x512)')
axes[0, 0].axis('off')

for idx, (scale, noise) in enumerate([(2, 'low'), (2, 'medium'), (2, 'high'), (4, 'low'), (4, 'medium')]):
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    deg = create_degradation_pipeline(scale, noise)(gt)
    axes[row, col].imshow(deg, cmap='gray', vmin=0, vmax=1)
    axes[row, col].set_title(f'Degraded {scale}x {noise} ({deg.shape[1]}x{deg.shape[0]})')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 3. Test Dataset & DataLoader

In [ ]:
# Test with synthetic data if real data not available
import tempfile
import shutil

with tempfile.TemporaryDirectory() as tmpdir:
    deg_dir = Path(tmpdir) / 'degraded'
    gt_dir = Path(tmpdir) / 'ground_truth'
    deg_dir.mkdir()
    gt_dir.mkdir()
    
    # Create dummy data
    for i in range(20):
        gt = np.random.rand(512, 512).astype(np.float32)
        # Add some structure
        cv2.line(gt, (0, 256), (512, 256), 1.0, 1)
        cv2.circle(gt, (256, 256), 50, 0.5, -1)
        gt = cv2.GaussianBlur(gt, (3, 3), 0.5)
        
        deg = cv2.resize(gt, (256, 256), interpolation=cv2.INTER_CUBIC)
        deg = np.clip(deg + np.random.randn(*deg.shape) * 0.1, 0, 1)
        
        cv2.imwrite(str(gt_dir / f'img_{i:04d}.png'), (gt * 255).astype(np.uint8))
        cv2.imwrite(str(deg_dir / f'img_{i:04d}.png'), (deg * 255).astype(np.uint8))
    
    # Test dataset
    dataset = SemiconductorDataset(tmpdir, patch_size=256, scale=2)
    print(f"Dataset size: {len(dataset)}")
    
    sample = dataset[0]
    print(f"Degraded: {sample['degraded'].shape}, range=[{sample['degraded'].min():.3f}, {sample['degraded'].max():.3f}]")
    print(f"Ground Truth: {sample['ground_truth'].shape}, range=[{sample['ground_truth'].min():.3f}, {sample['ground_truth'].max():.3f}]")
    
    # Test dataloader
    train_loader, val_loader = create_dataloaders(tmpdir, batch_size=4, num_workers=0)
    batch = next(iter(train_loader))
    print(f"\nBatch degraded: {batch['degraded'].shape}")
    print(f"Batch ground_truth: {batch['ground_truth'].shape}")

## 4. Visualize Augmentations

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    deg_dir = Path(tmpdir) / 'degraded'
    gt_dir = Path(tmpdir) / 'ground_truth'
    deg_dir.mkdir()
    gt_dir.mkdir()
    
    gt = np.zeros((512, 512), dtype=np.float32)
    cv2.rectangle(gt, (150, 150), (350, 350), 1.0, -1)
    cv2.circle(gt, (256, 256), 80, 0.5, -1)
    gt = cv2.GaussianBlur(gt, (5, 5), 1.0)
    deg = cv2.resize(gt, (256, 256), interpolation=cv2.INTER_CUBIC)
    deg = np.clip(deg + np.random.randn(*deg.shape) * 0.1, 0, 1)
    
    cv2.imwrite(str(gt_dir / 'test.png'), (gt * 255).astype(np.uint8))
    cv2.imwrite(str(deg_dir / 'test.png'), (deg * 255).astype(np.uint8))
    
    dataset = SemiconductorDataset(tmpdir, patch_size=256, scale=2, augment=True)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i in range(12):
        sample = dataset[0]  # Same sample, different augmentations
        row = i // 4
        col = i % 4
        if col < 2:
            img = sample['degraded'][0].numpy()
            title = 'Degraded'
        else:
            img = sample['ground_truth'][0].numpy()
            title = 'Ground Truth'
        axes[row, col].imshow(img, cmap='gray', vmin=-1, vmax=1)
        axes[row, col].set_title(f'{title} (aug {i})')
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Test Model Forward Pass

In [ ]:
import torch
from models import create_model

# Create model
model = create_model('nafnet', scale=2)
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Test forward
x = torch.randn(1, 1, 256, 256)
y = model(x)
print(f"Input: {x.shape} -> Output: {y.shape}")

# Test 4x
model4 = create_model('nafnet', scale=4)
x4 = torch.randn(1, 1, 128, 128)
y4 = model4(x4)
print(f"Input: {x4.shape} -> Output: {y4.shape}")

## 6. Test Loss Function

In [ ]:
from models import create_loss

loss_fn = create_loss({})
pred = torch.randn(2, 1, 256, 256)
target = torch.randn(2, 1, 256, 256)
losses = loss_fn(pred, target)

for k, v in losses.items():
    print(f"{k}: {v.item():.4f}")